## Data Loading

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("Delivery_Logistics.csv")

In [4]:
df.head()

,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost
0,250.99,delhivery,automobile parts,bike,same day,west,clear,297.0,46.96,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000008,no,delivered,3,1632.7206
1,250.99,xpressbees,cosmetics,ev van,express,central,cold,89.6,47.39,1970-01-01 00:00:00.000000002,1970-01-01 00:00:00.000000003,no,delivered,5,640.1700
2,250.99,shadowfax,groceries,truck,two day,east,rainy,273.5,26.89,1970-01-01 00:00:00.000000010,1970-01-01 00:00:00.000000016,no,delivered,4,1448.1700
3,250.99,dhl,electronics,ev van,same day,east,cold,269.7,12.69,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000008,no,delivered,3,1486.5700
4,250.99,dhl,clothing,van,two day,north,foggy,256.7,37.02,1970-01-01 00:00:00.000000009,1970-01-01 00:00:00.000000016,no,delivered,4,1394.5600


## Data Inspection

In [5]:
df.shape   # Dataset Size

(25000, 15)

In [6]:
df.columns

Index(['delivery_id', 'delivery_partner', 'package_type', 'vehicle_type',
       'delivery_mode', 'region', 'weather_condition', 'distance_km',
       'package_weight_kg', 'delivery_time_hours', 'expected_time_hours',
       'delayed', 'delivery_status', 'delivery_rating', 'delivery_cost'],
      dtype='object')

In [7]:
df.info() # Data Types

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   delivery_id          25000 non-null  float64
 1   delivery_partner     25000 non-null  object 
 2   package_type         25000 non-null  object 
 3   vehicle_type         25000 non-null  object 
 4   delivery_mode        25000 non-null  object 
 5   region               25000 non-null  object 
 6   weather_condition    25000 non-null  object 
 7   distance_km          25000 non-null  float64
 8   package_weight_kg    25000 non-null  float64
 9   delivery_time_hours  25000 non-null  object 
 10  expected_time_hours  25000 non-null  object 
 11  delayed              25000 non-null  object 
 12  delivery_status      25000 non-null  object 
 13  delivery_rating      25000 non-null  int64  
 14  delivery_cost        25000 non-null  float64
dtypes: float64(4), int64(1), object(10)


In [8]:
df.isnull().sum()  # Missing values

delivery_id            0
delivery_partner       0
package_type           0
vehicle_type           0
delivery_mode          0
region                 0
weather_condition      0
distance_km            0
package_weight_kg      0
delivery_time_hours    0
expected_time_hours    0
delayed                0
delivery_status        0
delivery_rating        0
delivery_cost          0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df["delivery_id"].nunique()  # Unique Delivery IDs

24502

## Data Cleaning


#### Inspect Time Columns

In [11]:
df[["delivery_time_hours", "expected_time_hours"]].head(10)  

,delivery_time_hours,expected_time_hours
0,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000008
1,1970-01-01 00:00:00.000000002,1970-01-01 00:00:00.000000003
2,1970-01-01 00:00:00.000000010,1970-01-01 00:00:00.000000016
3,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000008
4,1970-01-01 00:00:00.000000009,1970-01-01 00:00:00.000000016
5,1970-01-01 00:00:00.000000004,1970-01-01 00:00:00.000000002
6,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000008
7,1970-01-01 00:00:00.000000004,1970-01-01 00:00:00.000000008
8,1970-01-01 00:00:00.000000005,1970-01-01 00:00:00.000000008
9,1970-01-01 00:00:00.000000003,1970-01-01 00:00:00.000000008


#### Create Original Data Backup

In [12]:
df_raw = df.copy()

#### Convert Delivery and Expected Time

In [13]:
df["delivery_time_hours"] = (
    pd.to_datetime(df["delivery_time_hours"]).dt.nanosecond
)
df["expected_time_hours"] = (
    pd.to_datetime(df["expected_time_hours"]).dt.nanosecond
)

In [14]:
df[["delivery_time_hours", "expected_time_hours"]].head(10)  # Verify Converted Values

,delivery_time_hours,expected_time_hours
0,8,8
1,2,3
2,10,16
3,6,8
4,9,16
5,4,2
6,6,8
7,4,8
8,5,8
9,3,8


In [15]:
df[["delivery_time_hours", "expected_time_hours"]].dtypes

delivery_time_hours    int32
expected_time_hours    int32
dtype: object

#### Validate Time Data

In [16]:
df["delivery_time_hours"].describe()

count    25000.000000
mean         6.248040
std          3.140935
min          0.000000
25%          4.000000
50%          6.000000
75%          8.000000
max         19.000000
Name: delivery_time_hours, dtype: float64

#### Validate Expected Delivery Time

In [17]:
df["expected_time_hours"].describe()

count    25000.000000
mean        13.107680
std          7.559024
min          2.000000
25%          8.000000
50%          8.000000
75%         16.000000
max         24.000000
Name: expected_time_hours, dtype: float64

## Feature Engineering


#### Create Distance Category

In [18]:
df["distance_category"] = pd.cut(
    df["distance_km"],
    bins = [0,50,150,float("inf")],
    labels = ["Short", "Medium" , "Long"]
)

In [19]:
df[["distance_km", "distance_category"]].head(10)

,distance_km,distance_category
0,297.0,Long
1,89.6,Medium
2,273.5,Long
3,269.7,Long
4,256.7,Long
5,48.4,Short
6,198.3,Long
7,114.6,Medium
8,142.4,Medium
9,47.1,Short


#### Create Cost per Kilometer

In [20]:
df["cost_per_km"] = df["delivery_cost"] / df["distance_km"]

In [21]:
df[["delivery_cost", "distance_km", "cost_per_km"]].head(10)

,delivery_cost,distance_km,cost_per_km
0,1632.7206,297.0,5.497376
1,640.1700,89.6,7.144754
2,1448.1700,273.5,5.294954
3,1486.5700,269.7,5.511939
4,1394.5600,256.7,5.432645
5,391.4500,48.4,8.087810
6,1222.8700,198.3,6.166768
7,800.8900,114.6,6.988569
8,854.1800,142.4,5.998455
9,423.3400,47.1,8.988110


## Exploratory Data Analysis (EDA)


#### Delivery Status Distribution

In [22]:
df["delivery_status"].value_counts()

delivery_status
delivered    18331
delayed       5341
failed        1328
Name: count, dtype: int64

#### Delivery Status Percentage

In [23]:
df["delivery_status"].value_counts(normalize=True)*100

delivery_status
delivered    73.324
delayed      21.364
failed        5.312
Name: proportion, dtype: float64

#### Delivery Partner Performance

In [24]:
df["delivery_partner"].value_counts()

delivery_partner
xpressbees          2826
fedex               2818
dhl                 2802
ekart               2801
blue dart           2798
delhivery           2786
shadowfax           2736
ecom express        2722
amazon logistics    2711
Name: count, dtype: int64

#### Delivery Partner Delay Analysis

In [25]:
df.groupby("delivery_partner")["delayed"].apply(
    lambda x: (x=="yes").mean()*100
)

delivery_partner
amazon logistics    27.222427
blue dart           26.947820
delhivery           24.802584
dhl                 26.338330
ecom express        26.634827
ekart               27.418779
fedex               25.159688
shadowfax           27.302632
xpressbees          28.273178
Name: delayed, dtype: float64

In [26]:
df.groupby("vehicle_type")["delayed"].apply(
    lambda x: (x == "yes").mean()*100
)

vehicle_type
bike       27.043269
ev bike    26.434329
ev van     26.627794
scooter    26.137997
truck      27.044632
van        26.773346
Name: delayed, dtype: float64

#### Region-wise Delay Rate

In [27]:
df.groupby("region")["delayed"].apply(
    lambda x: (x == "yes").mean()*100
)

region
central    27.252964
east       25.797926
north      26.571024
south      26.783203
west       26.947988
Name: delayed, dtype: float64

In [28]:
df.groupby("weather_condition")["delayed"].apply(
    lambda x: (x == "yes").mean() * 100
)

weather_condition
clear     17.434530
cold      16.017316
foggy     30.315241
hot       17.118644
rainy     37.353153
stormy    41.448309
Name: delayed, dtype: float64

In [29]:
df.groupby("distance_category")["delayed"].apply(
    lambda x: (x == "yes").mean() * 100
)

C:\Users\Yash Mishra\AppData\Local\Temp\ipykernel_7496\3847644550.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("distance_category")["delayed"].apply(


distance_category
Short     17.020239
Medium    21.495552
Long      33.248549
Name: delayed, dtype: float64

In [30]:
df.groupby("delivery_rating")["delayed"].apply(
    lambda x: (x == "yes").mean() * 100
)

delivery_rating
1    100.000000
2    100.000000
3     37.190369
4      0.000000
5      0.000000
Name: delayed, dtype: float64

#### Delivery Cost Analysis

In [31]:
df["delivery_cost"].describe()

count    25000.000000
mean       864.944579
std        435.712593
min         95.667400
25%        490.800000
50%        867.535000
75%       1237.910000
max       1632.720600
Name: delivery_cost, dtype: float64

#### Distance vs Delivery Cost

In [32]:
  df.groupby("distance_category")["delivery_cost"].mean()

C:\Users\Yash Mishra\AppData\Local\Temp\ipykernel_7496\2998052189.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("distance_category")["delivery_cost"].mean()


distance_category
Short      239.635191
Medium     611.298061
Long      1236.475104
Name: delivery_cost, dtype: float64

In [34]:
df.to_csv("Delivery_Logistics_Cleaned.csv", index=False)

In [35]:
df.shape

(25000, 17)

In [36]:
df.columns.tolist()

['delivery_id',
 'delivery_partner',
 'package_type',
 'vehicle_type',
 'delivery_mode',
 'region',
 'weather_condition',
 'distance_km',
 'package_weight_kg',
 'delivery_time_hours',
 'expected_time_hours',
 'delayed',
 'delivery_status',
 'delivery_rating',
 'delivery_cost',
 'distance_category',
 'cost_per_km']